In [1]:
import pandas as pd
import os

# Veri yolunu kendi bilgisayarındaki yola göre düzelt
path = r"C:\Users\feyza\Desktop\uav_project\data\processed\aligned_flights"
files = os.listdir(path)
df = pd.read_csv(os.path.join(path, files[0]))

print("CSV içindeki kolon isimlerin:")
print(df.columns.tolist())

CSV içindeki kolon isimlerin:
['timestamp', 'gyro_rad[0]', 'gyro_rad[1]', 'gyro_rad[2]', 'accelerometer_m_s2[0]', 'accelerometer_m_s2[1]', 'accelerometer_m_s2[2]', 'x', 'y', 'z', 'vx', 'vy', 'vz']


In [2]:
import os
import pandas as pd

data_path = r"C:\Users\feyza\Desktop\uav_project\data\processed\aligned_flights"
files = [os.path.join(data_path, f) for f in os.listdir(data_path) if f.endswith('.csv')]

farkli_kolon_sayilari = {}

for f in files:
    df = pd.read_csv(f)
    n_cols = len(df.columns)
    if n_cols not in farkli_kolon_sayilari:
        farkli_kolon_sayilari[n_cols] = []
    farkli_kolon_sayilari[n_cols].append(os.path.basename(f))

print("Dosyaların kolon sayıları raporu:")
for count, filenames in farkli_kolon_sayilari.items():
    print(f"{count} kolonlu dosya sayısı: {len(filenames)}")
    if len(filenames) < 10: # Çok değilse isimlerini de yaz
        print(f"  Örnek: {filenames[0]}")
    

Dosyaların kolon sayıları raporu:
13 kolonlu dosya sayısı: 18
17 kolonlu dosya sayısı: 79


In [3]:
# 13 kolonlu dosyaları tespit edip taşıma
import shutil
import os

source_path = r"C:\Users\feyza\Desktop\uav_project\data\processed\aligned_flights"
backup_path = r"C:\Users\feyza\Desktop\uav_project\data\processed\backup_inconsistent"

if not os.path.exists(backup_path): os.makedirs(backup_path)

for f in os.listdir(source_path):
    file_path = os.path.join(source_path, f)
    df = pd.read_csv(file_path)
    if len(df.columns) < 15: # 17 olmayanları yakala
        shutil.move(file_path, os.path.join(backup_path, f))
        print(f"Taşındı: {f}")

Taşındı: master_flight_1.csv
Taşındı: master_flight_10.csv
Taşındı: master_flight_11.csv
Taşındı: master_flight_2.csv
Taşındı: master_flight_3.csv
Taşındı: master_flight_4.csv
Taşındı: master_flight_5.csv
Taşındı: master_flight_57.csv
Taşındı: master_flight_6.csv
Taşındı: master_flight_7.csv
Taşındı: master_flight_72.csv
Taşındı: master_flight_8.csv
Taşındı: master_flight_80.csv
Taşındı: master_flight_81.csv
Taşındı: master_flight_82.csv
Taşındı: master_flight_88.csv
Taşındı: master_flight_9.csv
Taşındı: master_flight_95.csv


In [9]:
import torch
# Sınıfını belleğe tekrar yükle (Model mimarisini tanısın)
class UAVHybridAutoencoder(torch.nn.Module):
    def __init__(self, input_dim=16, window_size=10):
        super(UAVHybridAutoencoder, self).__init__()
        self.cnn_encoder = torch.nn.Sequential(
            torch.nn.Conv1d(in_channels=window_size, out_channels=32, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.Conv1d(in_channels=32, out_channels=16, kernel_size=3, padding=1),
            torch.nn.ReLU()
        )
        self.lstm_encoder = torch.nn.LSTM(input_size=input_dim, hidden_size=32, num_layers=1, batch_first=True)
        self.lstm_decoder = torch.nn.LSTM(input_size=32, hidden_size=input_dim, num_layers=1, batch_first=True)
        self.cnn_decoder = torch.nn.Sequential(
            torch.nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.Conv1d(in_channels=32, out_channels=window_size, kernel_size=3, padding=1),
            torch.nn.Sigmoid()
        )
    def forward(self, x):
        cnn_out = self.cnn_encoder(x)
        lstm_out, _ = self.lstm_encoder(x)
        dec_lstm_out, _ = self.lstm_decoder(lstm_out)
        final_out = self.cnn_decoder(cnn_out)
        return final_out

# Modeli belleğe yükle
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UAVHybridAutoencoder(input_dim=16, window_size=10).to(DEVICE)
model.load_state_dict(torch.load(r"C:\Users\feyza\Desktop\uav_project\models\hybrid_model.pth", map_location=DEVICE))
model.eval()
print("✅ Model belleğe yüklendi!")

✅ Model belleğe yüklendi!


C:\Users\feyza\AppData\Local\Temp\ipykernel_12056\829345947.py:30: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(r"C:\Users\feyza\Desktop\ua

In [10]:
import os
import pandas as pd
import numpy as np
import torch
import pickle # İşte bu satır eksikti, ekledik!
# 1. Klasördeki dosyaları listeleyelim
aligned_dir = r"C:\Users\feyza\Desktop\uav_project\data\processed\aligned_flights"
mevcut_dosyalar = [f for f in os.listdir(aligned_dir) if f.endswith('.csv')]

# 2. İçerideki ilk dosyayı seçelim (Taşıdıklarımızı değil, kalanları alır)
test_dosya_yolu = os.path.join(aligned_dir, mevcut_dosyalar[0])
df_test = pd.read_csv(test_dosya_yolu)

# 3. Scaler'ı yükle
with open(r"C:\Users\feyza\Desktop\uav_project\models\global_scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

# 4. Veriyi işle
sensor_cols = list(scaler.feature_names_in_)
data = scaler.transform(df_test[sensor_cols].values)

# 5. Pencereleri oluştur
pencere = [data[0:10]] 
X_tensor = torch.tensor(np.array(pencere), dtype=torch.float32).to(DEVICE)
print(f"✅ X_tensor başarıyla tanımlandı! Dosya: {mevcut_dosyalar[0]}")

✅ X_tensor başarıyla tanımlandı! Dosya: master_flight_12.csv


C:\Users\feyza\anaconda3\envs\uav_anomaly\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [12]:
from captum.attr import IntegratedGradients
import torch

# 1. ÖNEMLİ: MSE hatasını hesaplayan "Özel İleri Besleme" fonksiyonu
# Captum'un türev alabilmesi için tek bir skaler (MSE) döndüren bir fonksiyon yazıyoruz.
def forward_func(inputs):
    # Modelin tahminini al
    outputs = model(inputs)
    # MSE hatasını hesapla (Kayıp değeri ne kadar yüksekse, o sensör o kadar 'suçludur')
    mse = torch.mean((outputs - inputs) ** 2, dim=[1, 2])
    return mse

# 2. Captum'u bu yeni fonksiyonla başlatıyoruz
ig = IntegratedGradients(forward_func)

# 3. Veriyi hazırlıyoruz (requires_grad_=True türev alabilmek için şart!)
# X_tensor zaten tanımlıydı, onun ilk penceresini alıyoruz
test_window = X_tensor[0:1].to(DEVICE).requires_grad_(True)

# 4. Artık target belirtmemize gerek yok çünkü MSE tek bir skaler döndürüyor!
# attribution değerleri, 16 sensörün her birinin bu hatadaki "payını" gösterir.
attributions = ig.attribute(test_window)

# 5. Görselleştirme için anlamlı hale getirme
# attributions boyutu (1, 10, 16) -> (16) boyutuna indiriyoruz
sensor_scores = attributions.mean(dim=1).detach().cpu().numpy()[0]

print("✅ Sensör Katkı Puanları (Sensörlerin hata üzerindeki etkisi):")
for i, score in enumerate(sensor_scores):
    print(f"Sensör {i}: {score:.6f}")

✅ Sensör Katkı Puanları (Sensörlerin hata üzerindeki etkisi):
Sensör 0: -0.000097
Sensör 1: 0.000015
Sensör 2: -0.000004
Sensör 3: -0.000012
Sensör 4: 0.000011
Sensör 5: 0.000024
Sensör 6: 0.000008
Sensör 7: -0.000016
Sensör 8: -0.000001
Sensör 9: 0.000024
Sensör 10: -0.000022
Sensör 11: -0.000003
Sensör 12: -0.000001
Sensör 13: 0.000002
Sensör 14: 0.000006
Sensör 15: 0.000021


In [13]:
# Hangi sensörün en yüksek etkiye sahip olduğunu sayısal olarak görelim
attr_values = attributions.mean(dim=1).detach().cpu().numpy()[0]

# Sensör isimlerini ve etki puanlarını birleştir
df_analiz = pd.DataFrame({
    "Sensör": sensor_cols,
    "Etki_Skoru": np.abs(attr_values) # Pozitif/Negatif kafa karıştırmasın, şiddete bakıyoruz
})

# Büyükten küçüğe sırala
df_analiz = df_analiz.sort_values(by="Etki_Skoru", ascending=False)

print("🚨 En Çok Sapan 5 Sensör:")
print(df_analiz.head(5))

🚨 En Çok Sapan 5 Sensör:
                   Sensör  Etki_Skoru
0             gyro_rad[0]    0.000097
5   accelerometer_m_s2[2]    0.000024
9                      vx    0.000024
10                     vy    0.000022
15                   q[3]    0.000021


In [19]:
import os
import pandas as pd
import numpy as np
import torch
import pickle
from captum.attr import IntegratedGradients

# 1. AYARLAR (Yolları senin sistemine göre ayarladım)
ATTACK_POOL_DIR = r"C:\Users\feyza\Desktop\uav_project\data\processed\attack_master_pool"
MODELS_DIR = r"C:\Users\feyza\Desktop\uav_project\models"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. MODELİ YÜKLEME VE CAPTUM KURULUMU
# (Daha önce tanımladığın model sınıfı burada olmalı, 
# eğer hücrede tanımlı değilse en başa sınıf tanımını ekle)
# Tüm uçuş boyunca hata (MSE) değerlerine bak
with torch.no_grad():
    mse_all = torch.mean((model(X_t) - X_t) ** 2, dim=[1, 2]).cpu().numpy()

# En yüksek hata veren (yani saldırının en şiddetli olduğu) anı bul
saldiri_ani = np.argmax(mse_all) 

# Analizi o "saldırı anında" yap
pencere_saldiri = torch.tensor(np.array([data[saldiri_ani:saldiri_ani+10]]), ...).to(DEVICE)
attr = ig.attribute(pencere_saldiri)
# ... (Buradan sonra aynı imza analizini yap)
def load_model_for_analysis():
    with open(os.path.join(MODELS_DIR, "global_scaler.pkl"), "rb") as f:
        scaler = pickle.load(f)
    model = UAVHybridAutoencoder(input_dim=16, window_size=10).to(DEVICE)
    model.load_state_dict(torch.load(os.path.join(MODELS_DIR, "hybrid_model.pth"), map_location=DEVICE, weights_only=True))
    model.eval()
    return scaler, model

def forward_func(inputs, model):
    return torch.mean((model(inputs) - inputs) ** 2, dim=[1, 2])

scaler, model = load_model_for_analysis()
ig = IntegratedGradients(lambda x: forward_func(x, model))

# 3. İMZA ANALİZ FONKSİYONU
def analiz_et(kategori, dosya_adi):
    dosya_yolu = os.path.join(ATTACK_POOL_DIR, kategori, dosya_adi)
    df = pd.read_csv(dosya_yolu)
    sensor_cols = list(scaler.feature_names_in_)
    
    # Veriyi hazırla (ilk 10'lu pencereyi alıp analiz edelim)
    data = scaler.transform(df[sensor_cols].values)
    pencere = torch.tensor(np.array([data[0:10]]), dtype=torch.float32).to(DEVICE).requires_grad_(True)
    
    # Captum ile etki skorlarını al
    attr = ig.attribute(pencere)
    sensor_scores = np.abs(attr.mean(dim=1).detach().cpu().numpy()[0])
    
    # Tabloya dök
    df_analiz = pd.DataFrame({"Sensör": sensor_cols, "Etki_Skoru": sensor_scores})
    df_analiz = df_analiz.sort_values(by="Etki_Skoru", ascending=False)
    
    print(f"\n--- {kategori} / {dosya_adi} İMZA ANALİZİ ---")
    print(df_analiz.head(5))

analiz_et('Altitude','attack_master_2018-06-02_14_29_29.csv')
analiz_et('External_Position','attack_master_2018-05-24_19_57_23.csv')
analiz_et('Globol_Poasiton','attack_master_2018-07-03_18_57_00.csv')

# 4. TEST ETME (Buraları değiştirerek farklı dosyaları dene)
# Örnek: 'Altitude' klasöründeki bir dosyayı analiz et
# analiz_et('Altitude', 'master_flight_X.csv') 

# Örnek: 'External_Position' klasöründeki bir dosyayı analiz et
# analiz_et('External_Position', 'master_flight_Y.csv')

NameError: name 'X_t' is not defined

saldırı imzalarını bulmaya çalışıyoz

In [20]:
import torch
import torch.nn as nn

class UAVHybridAutoencoder(nn.Module):
    def __init__(self, input_dim=16, window_size=10):
        super(UAVHybridAutoencoder, self).__init__()
        self.cnn_encoder = nn.Sequential(
            nn.Conv1d(in_channels=window_size, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(in_channels=32, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.lstm_encoder = nn.LSTM(input_size=input_dim, hidden_size=32, num_layers=1, batch_first=True)
        self.lstm_decoder = nn.LSTM(input_size=32, hidden_size=input_dim, num_layers=1, batch_first=True)
        self.cnn_decoder = nn.Sequential(
            nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(in_channels=32, out_channels=window_size, kernel_size=3, padding=1),
            nn.Sigmoid()
        )
    def forward(self, x):
        cnn_out = self.cnn_encoder(x)
        lstm_out, _ = self.lstm_encoder(x) 
        dec_lstm_out, _ = self.lstm_decoder(lstm_out)
        final_out = self.cnn_decoder(cnn_out)
        return final_out

In [21]:
import os, pandas as pd, numpy as np, pickle
from captum.attr import IntegratedGradients

ATTACK_POOL_DIR = r"C:\Users\feyza\Desktop\uav_project\data\processed\attack_master_pool"
MODELS_DIR = r"C:\Users\feyza\Desktop\uav_project\models"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_model_for_analysis():
    with open(os.path.join(MODELS_DIR, "global_scaler.pkl"), "rb") as f:
        scaler = pickle.load(f)
    model = UAVHybridAutoencoder(input_dim=16, window_size=10).to(DEVICE)
    model.load_state_dict(torch.load(os.path.join(MODELS_DIR, "hybrid_model.pth"), map_location=DEVICE, weights_only=True))
    model.eval()
    return scaler, model

scaler, model = load_model_for_analysis()
ig = IntegratedGradients(lambda x: torch.mean((model(x) - x) ** 2, dim=[1, 2]))

In [22]:
def analiz_et_saldiri_ani(kategori, dosya_adi):
    dosya_yolu = os.path.join(ATTACK_POOL_DIR, kategori, dosya_adi)
    df = pd.read_csv(dosya_yolu)
    sensor_cols = list(scaler.feature_names_in_)
    data = scaler.transform(df[sensor_cols].values)
    
    # 1. Tüm uçuşu tensor yap (X_t)
    pencereler = [data[i:i+10] for i in range(len(data)-10+1)]
    X_t = torch.tensor(np.array(pencereler), dtype=torch.float32).to(DEVICE)
    
    # 2. En şiddetli saldırı anını bul
    with torch.no_grad():
        mse_all = torch.mean((model(X_t) - X_t) ** 2, dim=[1, 2]).cpu().numpy()
    saldiri_ani = np.argmax(mse_all)
    
    # 3. Analizi o an için yap
    pencere = torch.tensor(np.array([data[saldiri_ani:saldiri_ani+10]]), dtype=torch.float32).to(DEVICE).requires_grad_(True)
    attr = ig.attribute(pencere)
    sensor_scores = np.abs(attr.mean(dim=1).detach().cpu().numpy()[0])
    
    df_analiz = pd.DataFrame({"Sensör": sensor_cols, "Etki_Skoru": sensor_scores})
    print(f"\n--- {kategori} (Saldırı Anı Analizi) ---")
    print(df_analiz.sort_values(by="Etki_Skoru", ascending=False).head(5))

# Şimdi çalıştır
analiz_et_saldiri_ani('Altitude', 'attack_master_2018-06-02_14_29_29.csv')
analiz_et_saldiri_ani('External_Position', 'attack_master_2018-05-24_19_57_23.csv')

C:\Users\feyza\anaconda3\envs\uav_anomaly\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(



--- Altitude (Saldırı Anı Analizi) ---
         Sensör  Etki_Skoru
0   gyro_rad[0]    0.000099
8             z    0.000025
15         q[3]    0.000022
11           vz    0.000017
9            vx    0.000013

--- External_Position (Saldırı Anı Analizi) ---
         Sensör  Etki_Skoru
10           vy    0.001548
0   gyro_rad[0]    0.000096
9            vx    0.000025
15         q[3]    0.000024
8             z    0.000021


C:\Users\feyza\anaconda3\envs\uav_anomaly\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
